# 02 · Live-Cell Segmentation and Tracking
Segments macrophage nuclei/cytoplasm from live-cell zarr acquisitions (Live-1 and Live-2)  
using **Cellpose**, tracks across time with **Trackastra**, and appends Mtb channel  
intensity properties to each tracked object.  
Output: per-position `trackastra_labels` and a `tracks.csv` under each zarr store.

In [ ]:
import os
import zarr
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from ast import literal_eval
from natsort import natsorted
from tqdm.auto import tqdm
from numcodecs import Blosc
from cellpose import models
from trackastra.model import Trackastra
from trackastra.tracking import graph_to_ctc, graph_to_napari_tracks
import seaborn as sns
import napari

## 1 · Initialise models

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

cellpose_model = models.CellposeModel(gpu=(device == 'cuda'))
trackastra_model = Trackastra.from_pretrained('ctc', device=device)

# Channel indices within the zarr image array
MTB_CH   = 1
BF_CH    = -1
DPC_CH   = 0

## 2 · Segment and track all positions
Runs over Live-1 and Live-2 acquisitions.  
For each position: extracts BF + DPC channels, runs Cellpose segmentation,  
then Trackastra tracking, and writes labels back into the zarr store.

In [ ]:
base_dirs = [
    Path('/mnt/DATA3/BPP0050/BPP0050-1-Live-cell-to4i_Live-1__2025-04-09T18_25_04-Measurement 1/'),
    Path('/mnt/DATA3/BPP0050/BPP0050-1-Live-cell-to4i_Live-2__2025-04-10T18_45_48-Measurement 1/')
]

compressor = Blosc(cname='zstd', clevel=5, shuffle=Blosc.BITSHUFFLE)

for base_dir in base_dirs:
    zarr_dirs = natsorted((base_dir / 'acquisition/zarr').glob('*.zarr'))

    for zarr_dir in tqdm(zarr_dirs, desc=str(base_dir.name)):
        acq_ID = literal_eval(zarr_dir.stem)
        zarr_group = zarr.open_group(zarr_dir)

        bf  = zarr_group.images[:, BF_CH,  ...].max(axis=1)   # (T, Y, X)
        dpc = zarr_group.images[:, DPC_CH, ...].max(axis=1)   # (T, Y, X)
        segmentation_input = np.stack([bf, dpc], axis=1)       # (T, 2, Y, X)
        del bf, dpc

        # Cellpose segmentation
        masks, flows, styles = cellpose_model.eval(
            segmentation_input,
            diameter=100,
            channel_axis=1,
        )

        # Trackastra tracking
        track_graph = trackastra_model.track(masks, segmentation_input)
        masks_tracked, graph = graph_to_ctc(track_graph, masks, outdir=None)

        # Write segmentation labels to zarr
        label_path = zarr_dir / 'labels' / 'trackastra_labels' / '0'
        zarr.open_array(
            str(label_path), mode='w',
            shape=masks_tracked.shape,
            chunks=(1, 1024, 1024),
            dtype=masks_tracked.dtype,
            compressor=compressor
        )[:] = masks_tracked

## 3 · Export track tables to CSV
For each position, reads the Trackastra track table (with properties)
from the zarr store and exports it as a CSV for downstream analysis.

In [ ]:
for base_dir in base_dirs:
    zarr_dirs = natsorted((base_dir / 'acquisition/zarr').glob('*.zarr'))
    exp_name = 'live_1' if 'Live-1' in str(base_dir) else 'live_2'

    for zarr_dir in tqdm(zarr_dirs, desc=f'Mtb readout: {exp_name}'):
        acq_ID = literal_eval(zarr_dir.stem)
        pos_str = f'{acq_ID[0]}_{acq_ID[1]}'

        tracks_path = Path(
            f'/mnt/DATA3/BPP0050/{exp_name}.zarr/{pos_str}/labels/trackastra_labels/tables/track/'
        )
        if not tracks_path.exists():
            continue

        track_columns = [f.name for f in tracks_path.iterdir() if f.is_dir()]
        track_data = {col: zarr.open(tracks_path / col, mode='r')[:] for col in track_columns}
        track_df = pd.DataFrame(track_data)

        csv_fn = tracks_path.parent.parent / 'tracks' / 'trackastra_wprops.csv'
        csv_fn.parent.mkdir(parents=True, exist_ok=True)
        track_df.to_csv(csv_fn, index=False)
        print(f'Saved: {csv_fn}')

## 4 · QC visualisation in napari
Preview images, segmentation, and tracks for a single position.

In [ ]:
import dask.array as da

pos = '3_3'
exp_name = 'live_1'
zarr_path = Path(f'/mnt/DATA3/BPP0050/{exp_name}.zarr')
zarr_root = zarr.open_group(str(zarr_path / pos), mode='r')

image_preview   = zarr_root['images']['0'].images[:48].max(axis=2)  # (T, C, Y, X)
seg_preview     = zarr_root['labels']['trackastra_labels']['0'][:48] # (T, Y, X)

tracks_path = zarr_path / pos / 'labels' / 'trackastra_labels' / 'tables' / 'track'
track_columns = [f.name for f in tracks_path.iterdir() if f.is_dir()]
track_df = pd.DataFrame({col: zarr.open(tracks_path / col, mode='r')[:] for col in track_columns})

track_subset = track_df[track_df['t'] < 48]
coords = track_subset[['track_id', 't', 'y', 'x']].values

viewer = napari.Viewer()
viewer.add_image(image_preview, channel_axis=1, name='Live images')
viewer.add_labels(seg_preview, name='Segmentation')
viewer.add_tracks(coords, name='Tracks')
napari.run()

## 5 · Track length QC

In [ ]:
track_lengths = track_df.groupby('track_id').size()

ax = sns.histplot(track_lengths, bins=30)
ax.figure.set_size_inches(6, 3)
sns.despine(offset=10)
ax.set(xlabel='Track length (frames)', ylabel='Count', title='Track length distribution')
print(f"Tracks > 100 frames: {(track_lengths > 100).sum()}")